<a href="https://colab.research.google.com/github/Adylbekovab/NN_Project/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Setup phase

In [38]:

#Installing Libraries
!pip install -q tensorflow torch torchvision opencv-python nltk transformers kaggle


!pip freeze > requirements.txt


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.12).
0 upgraded, 0 newly installed, 0 to remove and 20 not upgraded.


## Libraries installation tests

In [37]:
import tensorflow as tf
import torch
import cv2
import nltk
from transformers import pipeline

print("TensorFlow version:", tf.__version__)
print("PyTorch version:", torch.__version__)
print("OpenCV version:", cv2.__version__)

# Checking nltk
#nltk.download('punkt')

# Checking the Transformers
 #nlp = pipeline("sentiment-analysis")
#result = nlp("Image to Text")
#print(result)


try:
    nltk.download("punkt")  # Download tokenizer for NLP
    nlp = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
    result = nlp("Image to Text")
    print("✅ Transformers test successful:", result)
except Exception as e:
    print("⚠ Error checking Transformers:", e)

TensorFlow version: 2.18.0
PyTorch version: 2.5.1+cu124
OpenCV version: 4.11.0


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Device set to use cpu


✅ Transformers test successful: [{'label': 'POSITIVE', 'score': 0.993118166923523}]


### Working with Kaggle

In [40]:
import os
import json

kaggle_credentials = {
    "username": "your_kaggle_username",
    "key": "your_kaggle_api_key"
}

# Ensure the .kaggle directory exists
os.makedirs("/root/.kaggle", exist_ok=True)

# Save kaggle.json with credentials
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_credentials, f)

# Set correct permissions
os.chmod("/root/.kaggle/kaggle.json", 600)
print("✅ kaggle.json successfully created and configured!")



✅ kaggle.json successfully created and configured!


### Verify Kaggle API Access

In [5]:
!pip install kaggle --quiet
!kaggle datasets list


ref                                                               title                                              size  lastUpdated          downloadCount  voteCount  usabilityRating  
----------------------------------------------------------------  ------------------------------------------------  -----  -------------------  -------------  ---------  ---------------  
asinow/car-price-dataset                                          Car Price Dataset                                 135KB  2025-01-26 19:53:28          12939        189  1.0              
adilshamim8/sleep-cycle-and-productivity                          Sleep Cycle & Productivity                        155KB  2025-02-07 05:44:59           1736         34  1.0              
krishnanshverma/imdb-movies-dataset                               IMDb Movies Dataset                                38KB  2025-02-12 17:53:40            881         27  1.0              
mzohaibzeeshan/google-stock-price-data-2020-2025-googl      

### Redownload the dataset

In [7]:
!kaggle datasets download -d hsankesara/flickr-image-dataset --unzip -p ./flickr30k


Dataset URL: https://www.kaggle.com/datasets/hsankesara/flickr-image-dataset
License(s): CC0-1.0
100% 8.16G/8.16G [02:16<00:00, 140MB/s]
100% 8.16G/8.16G [02:16<00:00, 64.4MB/s]


## Data Selection and Processing for Flickr30k

In [31]:
import pandas as pd
import os
import json

# Define dataset path
#dataset_path = "./flickr30k"
json_path = os.path.join(dataset_path, "flickr30k", "dataset.json")

# Load captions from the correct annotation file
annotations_file = os.path.join(dataset_path, "annotations", "captions.txt")
captions_dict = {}

# Check if the file exists
if not os.path.exists(annotations_file):
    raise FileNotFoundError(f"The file {annotations_file} was not found. Please check the path.")

# Read captions file and store captions per image
with open(annotations_file, "r") as file:
    for line in file:
        image_id, caption = line.strip().split("\t")
        image_id = image_id.split("#")[0]  # Remove caption number (#0, #1, #2...)
        if image_id not in captions_dict:
            captions_dict[image_id] = []
        captions_dict[image_id].append(caption)

# Convert to DataFrame
df = pd.DataFrame(list(captions_dict.items()), columns=["image", "captions"])
print(df.head())


            image                                           captions
0  1000092795.jpg  [Two young guys with shaggy hair look at their...
1    10002456.jpg  [Several men in hard hats are operating a gian...
2  1000268201.jpg  [A child in a pink dress is climbing up a set ...
3  1000344755.jpg  [Someone in a blue shirt and hat is standing o...
4  1000366164.jpg  [Two men, one in a gray shirt, one in a black ...


In [34]:
# Check if the dataset annotations exist, otherwise download them
if not os.path.exists(json_path):
    print("🔄 Downloading annotation files...")
    !wget -q https://cs.stanford.edu/people/karpathy/deepimagesent/flickr30k.zip
    !unzip -q flickr30k.zip -d {dataset_path}
    print("Annotations downloaded!")

### Check dataset for Captions


In [28]:
import json

json_path = "./flickr30k/flickr30k/dataset.json"

# Load the JSON file
try:
    with open(json_path, "r") as file:
        data = json.load(file)
        print(" JSON annotations loaded successfully!")
except Exception as e:
    raise ValueError(f"⚠ Error loading JSON file: {e}")

# Print structure of JSON file
print(json.dumps(data, indent=4)[:1000])  # Show only first 1000 characters


✅ JSON annotations loaded successfully!
{
    "images": [
        {
            "sentids": [
                0,
                1,
                2,
                3,
                4
            ],
            "imgid": 0,
            "sentences": [
                {
                    "tokens": [
                        "two",
                        "young",
                        "guys",
                        "with",
                        "shaggy",
                        "hair",
                        "look",
                        "at",
                        "their",
                        "hands",
                        "while",
                        "hanging",
                        "out",
                        "in",
                        "the",
                        "yard"
                    ],
                    "raw": "Two young guys with shaggy hair look at their hands while hanging out in the yard.",
                    "imgid": 0,
                

### Extract Captions from dataset

In [47]:
import json
import os

json_path = "./flickr30k/flickr30k/dataset.json"
annotations_path = "./flickr30k/annotations"

# Create annotations directory if it doesn't exist
os.makedirs(annotations_path, exist_ok=True)

# Load JSON data
with open(json_path, "r") as file:
    data = json.load(file)

# Extract image-caption pairs
captions_file = os.path.join(annotations_path, "captions.txt")

with open(captions_file, "w") as f:
    for image in data["images"]:
        img_id = image["filename"]
        for sentence in image["sentences"]:
            caption = sentence["raw"]
            f.write(f"{img_id}\t{caption}\n")

print("Captions extracted and saved as captions.txt!")


✅ Captions extracted and saved as captions.txt!


### Update Script

In [25]:
annotations_file = "./flickr30k/annotations/captions.txt"


## Data Preprocessing

### 1. Load Necessary Libraries

In [41]:
import os
import json
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from tqdm import tqdm


### 2. Load Captions and Image Paths

In [42]:
# Define dataset paths
dataset_path = "./flickr30k"
annotations_path = os.path.join(dataset_path, "annotations", "captions.txt")

# Load image captions
captions_dict = {}
with open(annotations_path, "r") as file:
    for line in file:
        image_id, caption = line.strip().split("\t")
        image_id = image_id.split("#")[0]  # Remove # caption index
        if image_id not in captions_dict:
            captions_dict[image_id] = []
        captions_dict[image_id].append(caption)

# Convert to DataFrame
df = pd.DataFrame(list(captions_dict.items()), columns=["image", "captions"])
print("Captions loaded:", df.shape)


✅ Captions loaded: (31014, 2)


### 3. Split Data into Train/Val/Test Sets

In [45]:
# Train/Val/Test Split (80/10/10)
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(test_data, test_size=0.5, random_state=42)

print(f" Data split: Train {len(train_data)}, Val {len(val_data)}, Test {len(test_data)}")


 Data split: Train 24811, Val 3101, Test 3102


### 4. Tokenize Captions

In [44]:
# Flatten captions list
all_captions = [caption for captions in df["captions"] for caption in captions]

# Tokenizer settings
tokenizer = Tokenizer(num_words=5000, oov_token="<UNK>", filters='!"#$%&()*+.,-/:;=?@[]^_`{|}~')
tokenizer.fit_on_texts(all_captions)

# Convert captions to sequences
max_length = max(len(caption.split()) for caption in all_captions)
df["tokenized_captions"] = df["captions"].apply(lambda captions:
    pad_sequences(tokenizer.texts_to_sequences(captions), maxlen=max_length, padding="post"))

print(f" Tokenization complete. Vocabulary size: {len(tokenizer.word_index)}")


 Tokenization complete. Vocabulary size: 18659


### 5: Process Images (Resize & Normalize)

In [50]:
import numpy as np

def preprocess_image(image_path, target_size=(224, 224)):
    """Preprocess an image: resize, normalize, and handle missing files."""

    if not os.path.exists(image_path):
        print(f"⚠ Image {image_path} not found. It might be added later.")
        return None  # Returning None instead of an empty array

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, target_size)
    image = image / 255.0  # Normalize pixel values
    return image


In [51]:
missing_images = []  # List to store missing image filenames

df["processed_images"] = df["image"].apply(lambda img:
    preprocess_image(os.path.join(dataset_path, "images", img)) if img not in missing_images else None
)

# Save missing files list
with open("missing_images.txt", "w") as f:
    for img in missing_images:
        f.write(img + "\n")

print(f"⚠ {len(missing_images)} images are missing. They are saved in missing_images.txt")


Streaming output truncated to the last 5000 lines.
⚠ Image ./flickr30k/images/533508800.jpg not found. It might be added later.
⚠ Image ./flickr30k/images/5335780437.jpg not found. It might be added later.
⚠ Image ./flickr30k/images/533601247.jpg not found. It might be added later.
⚠ Image ./flickr30k/images/5336013973.jpg not found. It might be added later.
⚠ Image ./flickr30k/images/533602654.jpg not found. It might be added later.
⚠ Image ./flickr30k/images/5336798287.jpg not found. It might be added later.
⚠ Image ./flickr30k/images/5337047043.jpg not found. It might be added later.
⚠ Image ./flickr30k/images/5337563702.jpg not found. It might be added later.
⚠ Image ./flickr30k/images/533854547.jpg not found. It might be added later.
⚠ Image ./flickr30k/images/5338568818.jpg not found. It might be added later.
⚠ Image ./flickr30k/images/5338988058.jpg not found. It might be added later.
⚠ Image ./flickr30k/images/533979933.jpg not found. It might be added later.
⚠ Image ./flickr30

In [52]:
missing_images = []  # List to store missing image filenames

df["processed_images"] = df["image"].apply(lambda img:
    preprocess_image(os.path.join(dataset_path, "images", img)) if img not in missing_images else None
)

# Save missing files list
with open("missing_images.txt", "w") as f:
    for img in missing_images:
        f.write(img + "\n")

print(f"⚠ {len(missing_images)} images are missing. They are saved in missing_images.txt")
#If a file is missing → it is written to missing_images.txt.
#When the file appears → it can be automatically re-processed.
#This avoids errors and allows you to work with a dynamically updated dataset


Streaming output truncated to the last 5000 lines.
⚠ Image ./flickr30k/images/533508800.jpg not found. It might be added later.
⚠ Image ./flickr30k/images/5335780437.jpg not found. It might be added later.
⚠ Image ./flickr30k/images/533601247.jpg not found. It might be added later.
⚠ Image ./flickr30k/images/5336013973.jpg not found. It might be added later.
⚠ Image ./flickr30k/images/533602654.jpg not found. It might be added later.
⚠ Image ./flickr30k/images/5336798287.jpg not found. It might be added later.
⚠ Image ./flickr30k/images/5337047043.jpg not found. It might be added later.
⚠ Image ./flickr30k/images/5337563702.jpg not found. It might be added later.
⚠ Image ./flickr30k/images/533854547.jpg not found. It might be added later.
⚠ Image ./flickr30k/images/5338568818.jpg not found. It might be added later.
⚠ Image ./flickr30k/images/5338988058.jpg not found. It might be added later.
⚠ Image ./flickr30k/images/533979933.jpg not found. It might be added later.
⚠ Image ./flickr30